# Scientific Reports successor — Step 3

Parent-model continuity for the six published arteries. This notebook consumes a passing Step 2 completion gate and stops before perturbative or interaction-kernel calculations.

In [ ]:
from pathlib import Path
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'successor/scirep-waveform-susceptibility'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    repo_root = Path('/content/picoNewton')
    if not repo_root.exists():
        subprocess.run(['git','clone','https://github.com/khalid-saqr/picoNewton.git',str(repo_root)],check=True)
    subprocess.run(['git','-C',str(repo_root),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(repo_root),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)
    study_root = Path('/content/drive/MyDrive/picoNewton_susceptibility')
else:
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root/'picoNewton_v3').exists():
        repo_root = repo_root.parent
    study_root = repo_root/'piconewton_susceptibility_outputs'
print({'repo_root':str(repo_root),'study_root':str(study_root),'colab':IN_COLAB})

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'picoNewton_v3')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'piconewton_susceptibility')],check=True)

## Step 2 gate

Step 3 refuses execution unless the prior completion gate is claim-bearing and authorizes Step 3.

In [ ]:
from piconewton_susceptibility.validation import validate_bootstrap_artifacts
step2_root = study_root/'bootstrap'/'step2'
step2 = validate_bootstrap_artifacts(step2_root, require_claim_bearing=True, expected_storage_mode='drive' if IN_COLAB else None)
assert step2['passed'], step2
step2

## Publication-resolution parent continuity

In [ ]:
from piconewton_susceptibility.continuity import Step3Config, run_parent_continuity
step3_root = study_root/'step3_parent_continuity'
result = run_parent_continuity(step3_root, step2_root, Step3Config())
manifest = result['manifest']
assert manifest['status'] == 'complete', manifest
assert manifest['allowed_next_step'] == 4, manifest
manifest

In [ ]:
display(result['summary'][['artery_id','alpha_computed','anisotropic_signed_rms_n','isotropic_signed_rms_n','anisotropic_signed_excess_rms_n','signed_excess_fraction_of_isotropic_rms']])
display(result['convergence'].groupby('dimension')[['max_total_relative_change','max_excess_relative_change']].max())

## Boundary assertion

No perturbative hierarchy, interaction kernel, susceptibility functional, crossed matrix, or critical-anisotropy inversion is run in Step 3.

In [ ]:
assert result['manifest']['gates']['perturbation_kernel_or_inversion_run'] is False
print('STEP 3 COMPLETE — parent continuity only')